# 01_simple_agent_and_react: Observe-Think-Act Loop from Scratch

This notebook builds a simple ReAct agent loop from scratch in pure Python. It utilizes the Tavily Search API dynamically to gather live search observations, and uses OpenAI Chat models (`gpt-4o-mini`) to reason, execute, and summarize research queries.

### ReAct Paradigm Math
The ReAct loop integrates reasoning and acting dynamically:
$$\text{Query} \to \text{Thought}_1 \to \text{Action}_1 \to \text{Observation}_1 \to \text{Thought}_2 \to \text{Action}_2 \dots \to \text{Final Answer}$$
At each step, the agent evaluates the user goal, executes a tool (search query), and updates the context before synthesizing the final output, preventing static hallucinations.

In [1]:
import os
from dotenv import load_dotenv
from tavily import TavilyClient

# 1. Load keys from root .env
load_dotenv(dotenv_path=r"d:\\Study\\Prep\\.env")
tavily_key = os.getenv("TAVILY_API_KEY")
tavily_client = TavilyClient(api_key=tavily_key) if tavily_key else None
print("Tavily client initialized:", tavily_client is not None)

Tavily client initialized: True


In [2]:
# 2. ReAct Agent Loop implementation
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Custom Search Tool
def web_search(query: str) -> str:
    if tavily_client:
        try:
            results = tavily_client.search(query=query, max_results=2)
            return "\n".join([f"Source: {r['url']}\nSnippet: {r['content']}" for r in results['results']])
        except Exception as e:
            return f"Tavily API failed: {e}. Fallback: AI Agents are defined as autonomous systems."
    return "No Tavily API key found. Fallback: AI Agents are defined as software entities acting autonomously on behalf of users."

goal = "Find the latest release version and features of the Gemini LLM family."
print("Objective:", goal)

# Turn 1
thought_1 = "I need to query the web to check the latest releases and updates of the Gemini LLM models."
print("\n--- Turn 1 ---")
print("Thought:", thought_1)
observation_1 = web_search("latest Gemini model releases features")
print("Observation:\n", observation_1[:300] + "...")

# Turn 2
thought_2 = "Now I will synthesize the search observations to write a comprehensive summary."
print("\n--- Turn 2 ---")
print("Thought:", thought_2)
prompt = f"Summarize the Gemini release features based on the context:\n{observation_1}\nQuery: {goal}"
response = llm.invoke(prompt).content
print("\nFinal Response:\n", response)

D:\Study\Prep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Objective: Find the latest release version and features of the Gemini LLM family.

--- Turn 1 ---
Thought: I need to query the web to check the latest releases and updates of the Gemini LLM models.


Observation:
 Source: https://gemini.google/release-notes
Snippet: What: Gemini is now powered by 2.0 Flash, our latest model that is designed for the agentic era, delivering fast responses and stronger performance across a number of key benchmarks for everyday help with tasks like brainstorming, learning, or wri...

--- Turn 2 ---
Thought: Now I will synthesize the search observations to write a comprehensive summary.



Final Response:
 The latest release in the Gemini LLM family is Gemini 2.0 Flash, announced on December 11, 2024. Key features include:

1. **2.0 Flash Model**: This model is designed for the agentic era, providing faster responses and enhanced performance across various benchmarks for tasks like brainstorming, learning, and writing.

2. **Gemini Advanced Access**: Users have access to a 1M token context window, allowing for up to 1,500 pages of file uploads, priority features like Deep Research and Gems, and 2TB of storage.

3. **Experimental Model**: The Gemini 2.0 Flash Experimental model offers a chat-optimized version with improvements in speed and academic benchmarks, though it may exhibit unexpected behaviors and compatibility issues.

4. **Video Generation**: The new Veo 3 model allows users to create 8-second videos with sound, featuring whimsical characters, background music, and sound effects, available to Google AI Ultra subscribers.

5. **Multimodal Live API**: This featu

### Output Explanation & Verification

#### Executed Results Trace:
- **Tavily Client Initialization**: Verifies if the Tavily search client is active.
- **Objective**: Finds the latest features and release versions of the Gemini LLM family.
- **Turn 1 (Thought & Action)**: Recognizes the query requires fresh web observations, querying Tavily to fetch model updates.
- **Turn 2 (Synthesis & Resolution)**: Summarizes the fetched Gemini model properties dynamically using `gpt-4o-mini` and outputs the final response.

This verifies the custom Python ReAct cycle, demonstrating how web search actions ground LLM synthesis with up-to-date facts.